# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Démontrer pourquoi la **moyenne arithmétique** est trompeuse sur des distributions asymétriques (délais logistiques, paniers e-commerce) et lui associer la **médiane (`PERCENTILE_APPROX`)** et l'**écart interquartile (`IQR`)**.
2. Maîtriser les **fonctions de fenêtrage SQL** (`OVER PARTITION BY`, `DENSE_RANK`, sommes cumulées de Pareto, `LAG` mensuel).
3. Alimenter la **Couche Gold** (`/lakehouse/gold/{kpi_categories, kpi_states, kpi_monthly}`) directement lue par notre application **Streamlit** (`http://localhost:8501`).
4. Comprendre le **Small Files Problem** sur HDFS (blocs de 16 Mo) et compacter les tables Gold.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# Enregistrement d'une vue temporaire SQL propre sur les commandes livrées de la couche Silver
spark.sql("""
    CREATE OR REPLACE TEMP VIEW v_delivered_orders AS
    SELECT *
    FROM delta.`/lakehouse/silver/orders`
    WHERE order_status = 'delivered'
      AND delivery_days IS NOT NULL
""")

print("Commandes livrées exploitables :", spark.sql("SELECT COUNT(*) FROM v_delivered_orders").collect()[0][0])

## 1. Moyenne vs Médiane : Preuve Statistique de l'Asymétrie Logistique

En logistique e-commerce, une livraison ne peut pas durer moins de 0 jour, mais des grèves de transporteurs ou des zones amazoniennes isolées génèrent une **longue traîne à droite** (commandes livrées en 60 à 150+ jours).
Vérifions le coefficient d'asymétrie (*skewness*) et comparons moyenne et quantiles globaux :

In [ ]:
spark.sql("""
    SELECT
        COUNT(*)                                         AS nb_orders,
        ROUND(AVG(delivery_days), 2)                     AS mean_days,
        ROUND(STDDEV(delivery_days), 2)                  AS std_days,
        ROUND(SKEWNESS(delivery_days), 2)                AS skewness,
        ROUND(PERCENTILE_APPROX(delivery_days, 0.25), 2) AS p25_days,
        ROUND(PERCENTILE_APPROX(delivery_days, 0.50), 2) AS median_days,
        ROUND(PERCENTILE_APPROX(delivery_days, 0.75), 2) AS p75_days,
        ROUND(PERCENTILE_APPROX(delivery_days, 0.95), 2) AS p95_days,
        ROUND(MAX(delivery_days), 2)                     AS max_days
    FROM v_delivered_orders
""").show(truncate=False)

> **Interprétation statistique :**
> - Un `skewness > 1` traduit une forte asymétrie positive : la **moyenne** (`mean_days`) est systématiquement tirée vers le haut par les 5 % de livraisons extrêmes (`p95_days`, `max_days`).
> - Pour engager un SLA client réaliste, le couple **Médiane (`P50`) + IQR (`P75 - P25`)** (ou le quantile `P90`) est beaucoup plus robuste que `Moyenne ± Écart-type`.
> - *Note algorithmique Big Data :* En environnement distribué, `PERCENTILE_APPROX` utilise un croquis probabiliste compact (*Greenwald-Khanna / quantile digest*) évitant de trier globalement toutes les lignes sur une seule partition !

---
## 2. Table Gold #1 : Performance par Catégorie (`/lakehouse/gold/kpi_categories`)

Nous enrichissons la requête de référence du projet avec une **loi de Pareto** (part cumulée du chiffre d'affaires via fenêtre `SUM(...) OVER (ORDER BY total_revenue DESC)`).

In [ ]:
df_gold_categories = spark.sql("""
    WITH cat_agg AS (
        SELECT
            category,
            COUNT(*)                                         AS total_orders,
            ROUND(SUM(total_amount), 2)                      AS total_revenue,
            ROUND(AVG(total_amount), 2)                      AS mean_basket,
            ROUND(PERCENTILE_APPROX(total_amount, 0.50), 2)  AS median_basket,
            ROUND(AVG(delivery_days), 2)                     AS mean_delivery_days,
            ROUND(PERCENTILE_APPROX(delivery_days, 0.50), 2) AS median_delivery_days,
            ROUND(
                PERCENTILE_APPROX(delivery_days, 0.75) - PERCENTILE_APPROX(delivery_days, 0.25),
                2
            )                                                AS iqr_delivery_days,
            ROUND(100.0 * AVG(is_late), 2)                   AS late_rate_pct,
            ROUND(AVG(review_score), 2)                      AS avg_review_score
        FROM v_delivered_orders
        GROUP BY category
        HAVING COUNT(*) >= 20
    )
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        ROUND(
            100.0 * SUM(total_revenue) OVER (
                ORDER BY total_revenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) / SUM(total_revenue) OVER (),
            2
        ) AS cumulative_revenue_pct
    FROM cat_agg
    ORDER BY total_orders DESC
""")

(
    df_gold_categories
    .coalesce(1)  # Table Gold agrégée (< 100 lignes) -> 1 seul fichier Parquet propre (évite le Small Files Problem)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/lakehouse/gold/kpi_categories")
)

df_gold_categories.show(12, truncate=False)

## 3. Table Gold #2 : Disparités Géographiques par État (`/lakehouse/gold/kpi_states`)

Nous calculons pour chaque État brésilien (`customer_state`) les indicateurs logistiques ainsi que la **catégorie n°1** locale grâce à `ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY COUNT(*) DESC)`.

In [ ]:
df_gold_states = spark.sql("""
    WITH state_cat_rank AS (
        SELECT
            customer_state,
            category AS top_category,
            ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY COUNT(*) DESC) AS rn
        FROM v_delivered_orders
        WHERE customer_state IS NOT NULL
        GROUP BY customer_state, category
    ),
    state_metrics AS (
        SELECT
            customer_state,
            COUNT(*)                                         AS total_orders,
            ROUND(SUM(total_amount), 2)                      AS total_revenue,
            ROUND(AVG(freight_amount / NULLIF(total_amount, 0)) * 100.0, 2) AS freight_share_pct,
            ROUND(PERCENTILE_APPROX(delivery_days, 0.50), 2) AS median_delivery_days,
            ROUND(PERCENTILE_APPROX(delivery_days, 0.90), 2) AS p90_delivery_days,
            ROUND(100.0 * AVG(is_late), 2)                   AS late_rate_pct,
            ROUND(AVG(review_score), 2)                      AS avg_review_score
        FROM v_delivered_orders
        WHERE customer_state IS NOT NULL
        GROUP BY customer_state
    )
    SELECT
        m.*,
        r.top_category
    FROM state_metrics m
    LEFT JOIN state_cat_rank r
      ON m.customer_state = r.customer_state AND r.rn = 1
    ORDER BY total_orders DESC
""")

(
    df_gold_states
    .coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/lakehouse/gold/kpi_states")
)

df_gold_states.show(10, truncate=False)

## 4. Table Gold #3 : Dynamique Mensuelle & Croissance MoM (`/lakehouse/gold/kpi_monthly`)

Utilisation de `LAG()` pour calculer la variation mensuelle du chiffre d'affaires (*Month-over-Month growth*) directement en SQL distribué :

In [ ]:
df_gold_monthly = spark.sql("""
    WITH monthly AS (
        SELECT
            order_month,
            COUNT(*)                                         AS total_orders,
            SUM(is_late)                                     AS late_orders,
            ROUND(100.0 * AVG(is_late), 2)                   AS late_rate_pct,
            ROUND(SUM(total_amount), 2)                      AS total_revenue,
            ROUND(PERCENTILE_APPROX(delivery_days, 0.50), 2) AS median_delivery_days,
            ROUND(AVG(review_score), 2)                      AS avg_review_score
        FROM v_delivered_orders
        WHERE order_month IS NOT NULL
        GROUP BY order_month
        HAVING COUNT(*) >= 50
    )
    SELECT
        *,
        ROUND(
            100.0 * (total_revenue - LAG(total_revenue, 1) OVER (ORDER BY order_month))
            / NULLIF(LAG(total_revenue, 1) OVER (ORDER BY order_month), 0),
            2
        ) AS revenue_mom_growth_pct
    FROM monthly
    ORDER BY order_month
""")

(
    df_gold_monthly
    .coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/lakehouse/gold/kpi_monthly")
)

df_gold_monthly.show(24, truncate=False)

## 5. Compaction Delta (`OPTIMIZE`) et Pourquoi Éviter le Sur-Partitionnement

Sur HDFS, chaque fichier (même de 4 Ko !) crée une entrée en RAM dans le **NameNode**. Si l'on partitionne naïvement une petite table par `(order_month, customer_state, category)`, Spark écrit des milliers de micro-fichiers de quelques kilo-octets (*Small Files Problem*).

Pour les grandes tables Silver subissant des `MERGE` fréquents, Delta Lake fournit la commande **`OPTIMIZE`** qui fusionne les petits fichiers Parquet en fichiers compacts :

In [ ]:
# Compaction des fichiers de la table Silver `orders` après nos opérations d'écriture / merge
spark.sql("OPTIMIZE delta.`/lakehouse/silver/orders`").show(truncate=False)

### Vérification immédiate sur Streamlit !
Ouvrez dès maintenant **http://localhost:8501** et cliquez sur le bouton **Actualiser le cache** : vos graphiques `kpi_categories` et `kpi_monthly` ainsi que l'explorateur `kpi_states` sont déjà en ligne !

Rendez-vous dans **`J4_RFM_Stats_Dashboard.ipynb`** pour les tests statistiques d'hypothèses et la segmentation clients RFM.